# Analyse NLP exploratoire du corpus

**Rôle : Étudiant 2 — thèmes, entités, événements, sentiment et évaluation.**

Ce notebook traite le corpus `corpus_final.csv`. Il utilise les titres pour tous les articles et le texte intégral uniquement lorsque `text_status` vaut `ok` ou `partial`. Il combine des embeddings pré-entraînés pour explorer des groupes sémantiques et une grille thématique interprétable. Les groupes automatiques sont des pistes à vérifier, pas des vérités annotées.

Le corpus contient 277 articles sur des sujets et périodes de collecte différents. Les volumes temporels ne représentent pas une mesure homogène de l'actualité.


## 1. Préparer l'environnement

Installer les dépendances avec `pip install -r requirements.txt`. Le premier chargement du modèle d'embeddings nécessite une connexion Internet et peut prendre quelques minutes. Modèle par défaut : `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`.

In [5]:
from pathlib import Path
import re, unicodedata, json, warnings
import numpy as np
import pandas as pd
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics.pairwise import cosine_similarity

ROOT = Path.cwd()
# Le notebook est situé dans notebooks/etudiant_2 ; remonter à la racine du dépôt.
if ROOT.name == "etudiant_2":
    ROOT = ROOT.parents[1]
CORPUS_PATH = ROOT / "data" / "raw" / "corpus_final_2.csv"
# En cas de lancement depuis un autre dossier, modifier cette variable manuellement.
OUT = ROOT / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)
print("Racine:", ROOT)
print("Corpus:", CORPUS_PATH)

Racine: /Users/dinameziane/Documents/2026:2027/veille-signaux-politiques/veille-media-france
Corpus: /Users/dinameziane/Documents/2026:2027/veille-signaux-politiques/veille-media-france/data/raw/corpus_final_2.csv


## 2. Charger et contrôler le corpus

Le fichier source mentionné dans le README peut être copié dans `data/raw/corpus_final.csv`. Le notebook conserve aussi `url`, `source` et les métadonnées utiles à la vérification des résultats.

In [6]:
if not CORPUS_PATH.exists():
    # Solution pratique pour le premier lancement : placer le CSV dans data/raw.
    raise FileNotFoundError(f"Copier corpus_final_2.csv dans {CORPUS_PATH}")
df = pd.read_csv(CORPUS_PATH, encoding="utf-8-sig")
required = {"article_id", "title", "published_at", "source", "url", "text", "text_status", "duplicate_of"}
missing = required - set(df.columns)
if missing: raise ValueError(f"Colonnes manquantes : {sorted(missing)}")
df["published_at"] = pd.to_datetime(df["published_at"], utc=True, errors="coerce")
df["text"] = df["text"].fillna("").astype(str)
df["title"] = df["title"].fillna("").astype(str)
df["source"] = df["source"].fillna("Source inconnue").astype(str)
# Le titre reste la base commune ; le texte est inclus seulement s'il a été récupéré.
usable_text = df["text_status"].isin(["ok", "partial"]) & df["text"].str.strip().ne("")
df["nlp_text"] = df["title"] + np.where(usable_text, "\n" + df["text"], "")
df["is_duplicate"] = df["duplicate_of"].fillna("").astype(str).str.strip().ne("")
print("Articles:", len(df), "| doublons signalés:", int(df.is_duplicate.sum()))
display(df["text_status"].value_counts(dropna=False).rename_axis("text_status").to_frame("articles"))
display(df[["published_at", "source", "title", "text_status"]].head())

Articles: 294 | doublons signalés: 1


,articles
text_status,
ok,139
summary,59
partial,53
error,43


,published_at,source,title,text_status
0,2026-10-12 03:30:17+00:00,Le Monde,Mouvement lycéen : face à une violence inédite...,partial
1,2026-10-12 06:21:12+00:00,CNews,Blocus des lycées : «LFI cherche à tester les ...,partial
2,2026-10-12 07:00:00+00:00,Le Bien Public,Plusieurs établissements de l'Université Bourg...,ok
3,2026-10-12 07:00:00+00:00,Sortir à Paris,Manifestation du 13 octobre : Pourquoi les étu...,ok
4,2026-10-12 08:20:00+00:00,La Provence,La mobilisation étudiante du 13 octobre espère...,ok


## 3. Explorer des thèmes émergents avec des embeddings

On calcule une représentation vectorielle de chaque titre (et du texte disponible), puis on regroupe les articles par proximité cosinus. Le seuil de distance hiérarchique est un paramètre exploratoire : examiner les groupes et ajuster `DISTANCE_THRESHOLD`. Ne pas confondre ces clusters sémantiques avec des catégories politiques validées. Avec un corpus petit et hétérogène, BERTopic peut être essayé ensuite, mais le résultat peut être instable.

In [7]:
from sentence_transformers import SentenceTransformer
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(MODEL_NAME)
texts = df["nlp_text"].tolist()
embeddings = model.encode(texts, batch_size=32, show_progress_bar=True, normalize_embeddings=True)
# Agglomératif à seuil de distance cosinus : pas besoin de choisir k.
DISTANCE_THRESHOLD = 0.55
try:
    clusterer = AgglomerativeClustering(n_clusters=None, metric="cosine", linkage="average", distance_threshold=DISTANCE_THRESHOLD)
except TypeError:  # Compatibilité sklearn plus ancien
    clusterer = AgglomerativeClustering(n_clusters=None, affinity="cosine", linkage="average", distance_threshold=DISTANCE_THRESHOLD)
df["semantic_cluster"] = clusterer.fit_predict(embeddings)
print("Groupes détectés:", df.semantic_cluster.nunique())

/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 10/10 [00:01<00:00,  6.59it/s]

Groupes détectés: 59


In [9]:
# Aperçu des groupes : lire les titres avant de donner un nom à chaque cluster.
for cid, group in df.loc[~df.is_duplicate].groupby("semantic_cluster"):
    print(f"\n### Groupe {cid} ({len(group)} articles)")
    for _, row in group.head(8).iterrows():
        print("-", row["published_at"].date() if pd.notna(row["published_at"]) else "date inconnue", "|", row["source"], "|", row["title"])


### Groupe 0 (63 articles)
- 2026-10-15 | CNews | Blocus des lycées : Sébastien Lecornu dénonce la transformation des manifestations en «mouvement de violences urbaines»
- 2026-10-16 | Le Figaro | Guillaume Tabard : «Convergence des luttes le 17 octobre, une menace réelle ou en trompe-l’œil ?»
- 2026-10-18 | franceinfo | Après les manifestations et les dégradations du 17 octobre, "les clés de l'apaisement sont dans les mains du gouvernement", selon la Fage
- 2026-10-18 | Orange Actualités | Tolbiac: la présidente de l’Unef estime que les dégradations sont aussi dues "à l’intervention des forces de l’ordre"
- 2026-10-18 | L'Union Étudiante | Communiqué de l’Union Étudiante suite à l’allocution d’Emmanuel Macron
- 2026-10-19 | Le Figaro | «Chaos», «casseurs déterminés», «pourrissement»… La classe politique condamne la violence des manifestations mais ne s’accorde pas sur les causes
- 2026-10-19 | Libération | 17 octobre : à Bordeaux, des politiques et des citoyens réunis pour «faire blo

## 4. Affecter une grille de thèmes politique et législative

Les thèmes sont initialisés à partir des sujets explicitement présents dans les titres du corpus (mobilisation étudiante, sécurité/maintien de l'ordre, textes et institutions, budget, éducation et précarité étudiante). Cette classification par mots-clés est une baseline explicable, pas un modèle sémantique. Une évolution utile consiste à nommer manuellement les clusters ci-dessus et enrichir `themes.yaml` à partir de ceux qui sont réellement observés.

In [10]:
THEMES = {
    "Mobilisations et mouvements sociaux": ["manifestation", "mobilisation", "blocus", "grève", "greve", "rassemblement", "syndicat", "lycéen", "lyceen", "étudiant", "etudiant"],
    "Maintien de l'ordre et sécurité": ["violence", "police", "policier", "sécurité", "securite", "casseur", "interpellation", "garde à vue", "garde a vue", "couvre-feu", "couvre feu", "interdiction", "maintien de l'ordre"],
    "Textes, lois et Parlement": ["projet de loi", "proposition de loi", "loi ", "texte", "sénat", "senat", "assemblée", "assemblee", "député", "depute", "commission", "adopte", "vote", "parlement"],
    "Budget et finances publiques": ["budget", "dépense", "depense", "déficit", "deficit", "financement", "milliard", "économie", "economie", "fiscal", "dette"],
    "Éducation et précarité étudiante": ["éducation", "education", "université", "universite", "parcoursup", "précarité étudiante", "precarite etudiante", "bourse", "étudiant", "etudiant", "lycée", "lycee"],
    "Vie politique et institutions": ["gouvernement", "ministre", "macron", "matignon", "elysée", "elysee", "parti", "président", "president", "élection", "election", "opposition"]
}
def normalize(s):
    s = unicodedata.normalize("NFKD", str(s).lower())
    return "".join(c for c in s if not unicodedata.combining(c))
def theme_scores(text):
    t = normalize(text)
    return {theme: sum(normalize(kw) in t for kw in kws) for theme, kws in THEMES.items()}
score_rows = [theme_scores(t) for t in df.nlp_text]
scores = pd.DataFrame(score_rows, index=df.index)

best_themes = scores.idxmax(axis=1)
best_scores = scores.max(axis=1)

df["theme"] = best_themes.where(best_scores.gt(0), "À examiner")

theme_kw_count = pd.Series({k: len(v) for k, v in THEMES.items()})

df["theme_score"] = (
    best_scores
    .div(best_themes.map(theme_kw_count))
    .where(best_scores.gt(0), 0)
    .clip(0, 1)
)

display(df["theme"].value_counts().to_frame("articles"))


,articles
theme,
Mobilisations et mouvements sociaux,94
À examiner,61
Vie politique et institutions,43
Éducation et précarité étudiante,35
"Textes, lois et Parlement",29
Maintien de l'ordre et sécurité,27
Budget et finances publiques,5


## 5. Extraire et normaliser les entités

Installer le modèle spaCy français `fr_core_news_md` si disponible. L'extraction NER est exploratoire, surtout sur les titres seuls ; les alias ci-dessous harmonisent quelques institutions et organisations fréquemment utiles. Vérifier les faux positifs dans les titres sources.

In [11]:
import spacy
try:
    nlp = spacy.load("fr_core_news_md")
except OSError:
    nlp = spacy.blank("fr")
    warnings.warn("fr_core_news_md absent : l'extraction NER sera vide. Voir README pour l'installation.")
ALIASES = {
    "assemblee nationale": "Assemblée nationale", "l'assemblee": "Assemblée nationale",
    "senat": "Sénat", "matignon": "Gouvernement (Matignon)", "elysee": "Présidence de la République",
    "gouvernement": "Gouvernement", "cgt": "CGT", "unef": "UNEF", "lfi": "La France insoumise",
    "la france insoumise": "La France insoumise", "rn": "Rassemblement national",
    "rassemblement national": "Rassemblement national", "cfdt": "CFDT"
}
def extract_entities(text):
    doc = nlp(str(text)[:100000])
    found = []
    for ent in doc.ents:
        if ent.label_ in {"PER", "PERSON", "ORG", "LOC", "GPE", "MISC"}:
            key = normalize(ent.text).strip(" .,'’\"")
            if len(key) > 1:
                found.append(ALIASES.get(key, ent.text.strip()))
    # Harmonisation d'alias dans le texte même, utile pour les institutions en NER manqué.
    norm_text = normalize(text)
    for alias, canonical in ALIASES.items():
        if re.search(r"\b" + re.escape(normalize(alias)) + r"\b", norm_text): found.append(canonical)
    return sorted(set(found))
df["entities"] = df.nlp_text.map(extract_entities).map(json.dumps)
display(df[["title", "entities"]].head(10))

,title,entities
0,Mouvement lycéen : face à une violence inédite...,"[""JavaScript is disabled in your browser"", ""Ja..."
1,Blocus des lycées : «LFI cherche à tester les ...,"[""Blocus des lyc\u00e9es"", ""CNEWS"", ""LR"", ""La ..."
2,Plusieurs établissements de l'Université Bourg...,"[""Activer JavaScript"", ""Adresse"", ""Chabot-Char..."
3,Manifestation du 13 octobre : Pourquoi les étu...,"[""CFDT"", ""CGT"", ""CGT \u00c9duc'action"", ""FCPE""..."
4,La mobilisation étudiante du 13 octobre espère...,"[""Aix"", ""Aix-Marseille universit\u00e9"", ""Aix-..."
5,Colère lycéenne et étudiante : quelques incide...,"[""Alsace"", ""Amadou"", ""Bas-Rhin"", ""Bischheim"", ..."
6,Blocus des lycées : la proviseure d'un établis...,"[""14e arrondissement de Paris"", ""@CNEWS"", ""@br..."
7,"Des manifestations dans de nombreuses villes, ...","[""AESH"", ""Ar\u00e8nes"", ""Association nationale..."
8,Mobilisation lycéenne : « Au moins huit person...,"[""JavaScript is disabled in your browser"", ""Ja..."
9,France | Les lycéens appellent à une nouvelle ...,"[""D\u00e9marr\u00e9"", ""Emmanuel Macron"", ""Fran..."


## 6. Regrouper des articles par événement

Un même événement est approché par similarité sémantique, proximité dans le temps (3 jours) et, si disponible, entité commune. Les doublons déjà signalés restent exclus des identifiants d'événement. Cette règle simple produit des pistes à vérifier et ne prétend pas résoudre les événements avec certitude.

In [12]:
# Union-find sur paires comparables ; un événement est un regroupement connexe.
n = len(df); parent = list(range(n))
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]; x = parent[x]
    return x
def union(a,b):
    ra, rb = find(a), find(b)
    if ra != rb: parent[rb] = ra
entity_sets = [set(json.loads(x)) for x in df.entities]
sim = cosine_similarity(embeddings)
EVENT_SIMILARITY = 0.68
for i in range(n):
    if df.iloc[i].is_duplicate: continue
    for j in range(i+1,n):
        if df.iloc[j].is_duplicate or pd.isna(df.iloc[i].published_at) or pd.isna(df.iloc[j].published_at): continue
        days = abs((df.iloc[i].published_at - df.iloc[j].published_at).total_seconds()) / 86400
        common_entity = bool(entity_sets[i] & entity_sets[j])
        if days <= 3 and sim[i,j] >= EVENT_SIMILARITY and (common_entity or sim[i,j] >= EVENT_SIMILARITY + 0.08): union(i,j)
roots = [find(i) for i in range(n)]
root_sizes = pd.Series(roots).value_counts()
df["event_id"] = [f"EVT-{r:04d}" if (not df.iloc[i].is_duplicate and root_sizes[r] > 1) else "" for i,r in enumerate(roots)]
print("Articles rattachés à un groupe événement:", int(df.event_id.ne("").sum()))

Articles rattachés à un groupe événement: 158


## 7. Sentiment lexical de démonstration

Ce score lexical est volontairement simple et doit être décrit comme une approximation. Il ne mesure ni la tonalité éditoriale, ni la polarité envers un acteur, et gère mal négation, ironie et citations. Ne pas réutiliser la colonne `sentiment` existante : le README du corpus indique qu'elle est renseignée seulement sur une partie et selon une méthode inconnue.

In [13]:
POS = {"accord", "soutien", "succès", "succes", "amélioration", "amelioration", "apaisement", "victoire", "adopte", "adopté", "espoir"}
NEG = {"violence", "crise", "colère", "colere", "échec", "echec", "fiasco", "blessé", "blesse", "interdit", "menace", "déficit", "deficit", "dégradation", "degradation", "répression", "repression"}
def lexical_sentiment(text):
    words = set(re.findall(r"[a-zà-ÿ]+", normalize(text)))
    pos = len(words & {normalize(x) for x in POS}); neg = len(words & {normalize(x) for x in NEG})
    if pos > neg: return "positif"
    if neg > pos: return "négatif"
    return "neutre"
df["sentiment_lexical"] = df.nlp_text.map(lexical_sentiment)

## 8. Évaluer sur un échantillon annoté manuellement

Renseigner `data/annotation/annotations.csv` avec les colonnes `article_id`, `theme_gold`, `is_relevant_gold`, `entities_gold`. Pour comparer les thèmes, conserver exactement les libellés produits par l'annotation. La feuille de route recommande 100–150 articles annotés et 20 annotés en commun par les trois personnes. Sans annotations, aucune précision/rappel/F1 ne doit être annoncée.

In [14]:
from sklearn.metrics import classification_report, accuracy_score
ann_path = ROOT / "data" / "annotation" / "annotations.csv"
if ann_path.exists():
    ann = pd.read_csv(ann_path)
    ev = df.merge(ann, on="article_id", how="inner")
    if {"theme_gold"}.issubset(ev.columns):
        mask = ev.theme_gold.notna() & ev.theme_gold.astype(str).str.strip().ne("")
        print("Évaluation thème (n =", int(mask.sum()), ")")
        print(classification_report(ev.loc[mask,"theme_gold"], ev.loc[mask,"theme"], zero_division=0))
    if "is_relevant_gold" in ev.columns:
        mask = ev.is_relevant_gold.notna()
        if mask.any():
            print("Pertinence : gold disponible, mais une règle dédiée doit être définie avant de comparer.")
else:
    print("Pas encore d'annotations : créer annotations.csv à partir du modèle fourni.")

Pas encore d'annotations : créer annotations.csv à partir du modèle fourni.


## 9. Exporter les résultats et relire des cas

La sortie respecte le contrat partagé du projet. Les entités sont sérialisées en JSON pour rester lisibles dans un CSV. Examiner un échantillon de lignes et les groupes avant intégration au pipeline/dashboard.

In [15]:
out_cols = ["article_id", "title", "published_at", "source", "url", "theme", "theme_score", "entities", "event_id", "sentiment_lexical", "semantic_cluster", "is_duplicate"]
result = df[out_cols].rename(columns={"sentiment_lexical":"sentiment", "is_duplicate":"duplicate_of_source"})
result.to_csv(OUT / "articles_nlp.csv", index=False, encoding="utf-8-sig")
print("Écrit:", OUT / "articles_nlp.csv")
display(result.sample(min(12, len(result)), random_state=42))

Écrit: /Users/dinameziane/Documents/2026:2027/veille-signaux-politiques/veille-media-france/data/processed/articles_nlp.csv


,article_id,title,published_at,source,url,theme,theme_score,entities,event_id,sentiment,semantic_cluster,duplicate_of_source
33,768b20098b9f698292945670b8a6ec8bb0018843,"Mobilisation lycéenne et étudiante: ""On n'atte...",2026-10-13 11:55:23+00:00,Orange Actualités,https://actu.orange.fr/videos/france/mobilisat...,Mobilisations et mouvements sociaux,0.545455,"[""BFMTV"", ""Ch\u00e8res lectrices"", ""Fermeture ...",EVT-0259,neutre,9,False
9,e73b06562f4a161e512b0aa365d9f0d907a5ddb6,France | Les lycéens appellent à une nouvelle ...,2026-10-12 12:22:42+00:00,La Presse,https://www.lapresse.ca/international/europe/2...,Mobilisations et mouvements sociaux,0.636364,"[""D\u00e9marr\u00e9"", ""Emmanuel Macron"", ""Fran...",EVT-0259,négatif,9,False
146,16da7bf8f25569122e962395294bf1e0c4aa2c97,David Lisnard fantasme sur «des camps d’entraî...,2026-10-20 09:59:41+00:00,Libération,https://www.liberation.fr/politique/david-lisn...,À examiner,0.000000,"[""David Lisnard"", ""Limousin""]",,neutre,54,False
282,32f1de7b597f9f2327b31f7379884fe8dc6a4d6c,Meeting du RN : la préfecture du Loiret interd...,2026-10-22 13:37:03+00:00,Libération,https://www.liberation.fr/politique/meeting-du...,Mobilisations et mouvements sociaux,0.090909,"[""Marine Le Pen"", ""Meeting du RN"", ""Orl\u00e9a...",,négatif,0,False
261,ecf8cd53ca809bbe9b186bbcfc1faa9f583c9930,Vente du FC Nantes. « Une bonne nouvelle » : G...,2026-10-22 11:10:52+00:00,Ouest-France,https://www.ouest-france.fr/sport/football/fc-...,À examiner,0.000000,"[""FC Nantes"", ""Gr\u00e9gory Poirier"", ""Phenix""]",,neutre,50,False
125,268a75c52c5e79ee18da04955bb6ee43476945e9,Qui sont vraiment les casseurs ultra-violents ...,2026-10-19 21:20:15+00:00,Le Figaro,https://lefigaro.fr/actualite-france/qui-sont-...,Budget et finances publiques,0.454545,"[""Astrolabe12"", ""Bartolom\u00e9 Lenoir"", ""CHRO...",,négatif,0,False
108,9259e081becf53f587819e12e831bb3ab5b002c9,Mobilisation en France: attention à ces images...,2026-10-19 15:20:44+00:00,RFI,https://www.rfi.fr/fr/france/20261019-mobilisa...,Mobilisations et mouvements sociaux,0.363636,"[""Au-del\u00e0 des images g\u00e9n\u00e9r\u00e...",EVT-0259,négatif,0,False
206,269d32d9856831337675722e81f5d4dd9d88112a,"Crise sociale en France: pris pour cible, Emma...",2026-10-21 20:39:49+00:00,RFI,https://www.rfi.fr/fr/france/20261021-crise-so...,Vie politique et institutions,0.750000,"[""Arabie saoudite"", ""Atlantique"", ""Bahre\u00ef...",EVT-0259,négatif,0,False
109,5c98657e377b70784e4fc83be628d333d9873f9c,"Chers casseurs, vous êtes les meilleurs agents...",2026-10-19 16:39:50+00:00,Libération,https://www.liberation.fr/idees-et-debats/opin...,Maintien de l'ordre et sécurité,0.076923,"[""Chers"", ""Marine Le Pen""]",,neutre,0,False
42,268c4b1cca7e7792d3fc1099a6273e6bff3ab7db,Mobilisation lycéenne. « La révolte d’une géné...,2026-10-13 17:02:36+00:00,Ouest-France,https://www.ouest-france.fr/education/mobilisa...,Mobilisations et mouvements sociaux,0.272727,[],,neutre,9,False


## 10. Interpréter les limites

- Les embeddings pré-entraînés comparent le sens des textes, mais ne garantissent pas que les clusters correspondent à des thèmes politiques cohérents.
- Le corpus est fortement hétérogène : mouvement étudiant du 12 au 18 octobre, puis flux RSS multi-sujets ; la collecte du 22 octobre est incomplète.
- Les articles sans texte restent analysables au titre seulement. Les résultats ne sont pas directement comparables à ceux utilisant un article intégral.
- NER et sentiment ont besoin d'une vérification humaine. Évaluer les thèmes et la pertinence sur l'échantillon annoté et documenter les erreurs.
- Les groupes d'événements sont sensibles aux seuils et peuvent fusionner des articles distincts ou séparer un même sujet.
- Les signaux d'affaires publiques doivent être définis en combinant accélération, diversité des médias et acteurs/institutions ; un cluster NLP isolé n'est pas un signal.
